# Amazon SageMaker Asynchronous Inference

In [1]:
import sagemaker
import boto3
from time import gmtime, strftime
from datetime import datetime
import time
import json
import os

boto_session = boto3.session.Session()
sm_session = sagemaker.session.Session()
sm_client = boto_session.client("sagemaker")
sm_runtime = boto_session.client("sagemaker-runtime")
region = boto_session.region_name
s3_client = boto3.client("s3", region)

today = datetime.now().strftime("%Y-%m-%d")

# Configuration
s3_bucket = 'textclassificationmldemo-model-archiving-us-east-1-2667'
bucket_prefix = 'models/model-a'
input_prefix = f'{bucket_prefix}/input/processed_json'
output_prefix = f'{bucket_prefix}/output/{today}'
print(input_prefix)

/opt/conda/lib/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
models/model-a/input/processed_json


## Create Model
Specifies the location of the pre-trained model stored in S3. 

In [2]:
from sagemaker import image_uris

model_s3_key = f"{s3_bucket}/{bucket_prefix}/model/model.tar.gz"
model_url = f"s3://{s3_bucket}/{model_s3_key}"
print(f"Uploading Model to {model_url}")

sm_role = sagemaker.get_execution_role()

# Specify an AWS container image and region as desired
container = image_uris.retrieve(region=region, framework="blazingtext", version="1")
print(container)

Uploading Model to s3://textclassificationmldemo-model-archiving-us-east-1-2667/textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/model/model.tar.gz


[03/03/25 22:16:01] INFO     Same images used for training and inference. Defaulting to image     ]8;id=241741;file:///opt/conda/lib/python3.11/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=986667;file:///opt/conda/lib/python3.11/site-packages/sagemaker/image_uris.py#393\393]8;;\
                             scope: inference.                                                                     

                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=976575;file:///opt/conda/lib/python3.11/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=369158;file:///opt/conda/lib/python3.11/site-packages/sagemaker/image_uris.py#530\530]8;;\

811284229777.dkr.ecr.us-east-1.amazonaws.com/blazingtext:1


#### Run this command if model is not available/created before

In [ ]:
# model_name = "TextClassification-ModelA"
# create_model_response = sm_client.create_model(
#     ModelName=model_name,
#     ExecutionRoleArn=sm_role,
#     PrimaryContainer={
#         "Image": container,
#         "ModelDataUrl": model_url,
#     },
# )

# print(f"Created Model: {create_model_response['ModelArn']}")

## Create EndpointConfig
Create the endpoint config if not created otherwise skip this cell. 

In [8]:
# Create EndpointConfig with async settings
endpoint_config_name = "TextClassificationMLDemo-TextClassification-Config"
model_name = "TextClassificationMLDemo-Model-A-20241218a-Model"

create_endpoint_config_response = sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "variant1",
            "ModelName": model_name,
            "InstanceType": "ml.m5.2xlarge",
            "InitialInstanceCount": 1,
        }
    ],
    AsyncInferenceConfig={
        "OutputConfig": {
            "S3OutputPath": f"s3://{s3_bucket}/{bucket_prefix}/{output_prefix}",
        },
        "ClientConfig": {"MaxConcurrentInvocationsPerInstance": 4},
    },
)
print(f"Created EndpointConfig: {create_endpoint_config_response['EndpointConfigArn']}")

Created EndpointConfig: arn:aws:sagemaker:us-east-1:266735847556:endpoint-config/TextClassificationMLDemo-TextClassification-Config


## Create Async Endpoint
Skip this if already endpoint is created.

In [9]:
# Create Async Endpoint using the same endpoint config name
endpoint_name = "Text-Classification-Model-Archiving"

create_endpoint_response = sm_client.create_endpoint(
    EndpointName=endpoint_name, 
    EndpointConfigName=endpoint_config_name  # use the same name here
)

print(f"Created Endpoint: {create_endpoint_response['EndpointArn']}")

Created Endpoint: arn:aws:sagemaker:us-east-1:266735847556:endpoint/Text-Classification-Model-Archiving


------

## Invoke Endpoint
Invoke endpoint with multiple json files.

In [12]:
endpoint_name = 'TextClassificationMLDemo-TextClassification-Endpoint'
# endpoint_name = 'Text-Classification-Model-Archiving'
INPUT_S3_PREFIX = f's3://{s3_bucket}/{input_prefix}/'
print(INPUT_S3_PREFIX)

def list_input_files_dynamic(s3_prefix):
    """
    Given an S3 prefix, list all S3 URIs for objects ending with '.json'
    in that folder using list_objects_v2 with pagination.
    """
    parts = s3_prefix.replace("s3://", "").split("/", 1)
    bucket = parts[0]
    key_prefix = parts[1] if len(parts) > 1 else ""
    if not key_prefix.endswith("/"):
        key_prefix += "/"
    
    file_list = []
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=key_prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith(".json"):
                file_uri = f"s3://{bucket}/{key}"
                file_list.append(file_uri)
                print(f"[{datetime.now()}] Found file: {file_uri}")
    return file_list

# print(list_input_files_dynamic(INPUT_S3_PREFIX))

s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/


In [8]:

def invoke_async_for_file(endpoint_name, input_file):
    """
    Invokes the asynchronous endpoint for a single input file.
    """
    print(f"[{datetime.now()}] Invoking async endpoint for file: {input_file}")
    response = sm_runtime.invoke_endpoint_async(
        EndpointName=endpoint_name,
        InputLocation=input_file,
        ContentType="application/jsonlines",  # each file contains a single JSON object
        Accept="application/jsonlines"
    )
    inference_id = response.get("InferenceId")
    print(f"[{datetime.now()}] InferenceId for {input_file}: {inference_id}")
    return inference_id


In [11]:
def main():
    input_files = list_input_files_dynamic(INPUT_S3_PREFIX)
    if not input_files:
        print(f"No input files found in {INPUT_S3_PREFIX}")
        return

    invocation_ids = {}
    start_time = time.time()
    
    for file in input_files:
        inference_id = invoke_async_for_file(endpoint_name, file)
        invocation_ids[file] = inference_id
        time.sleep(1)  # Optional delay between invocations

    elapsed = time.time() - start_time
    print(f"\nSubmitted {len(invocation_ids)} async requests in {elapsed:.2f} seconds.")
    print("The async endpoint will write outputs to the configured S3 output location.")

if __name__ == "__main__":
    main()

[2025-03-03 22:33:12.970881] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_1.json
[2025-03-03 22:33:12.971142] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_10.json
[2025-03-03 22:33:12.971180] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_11.json
[2025-03-03 22:33:12.971195] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_12.json
[2025-03-03 22:33:12.971208] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_13.json
[2025-03-03 22:33:12.971336] Found file: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/processed_json/input_14.json
[2025-03-03 22:33:12.971351] Found file: s3://textclassificationmldemo-model-archiving-us